[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList(
            [nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model),
            ) for i in range(num_experts)]
        )
        self.d_model = d_model
        self.d_ff = d_ff
        self.num_experts = num_experts
        self.top_k = top_k

    def forward(self, x):
      B, S, D = x.shape
      x_flat = x.reshape(-1, D)
      route = self.router(x_flat)
      topk_val, topk_idx = torch.topk(route, self.top_k ,dim=-1)
      topk_w = torch.softmax(topk_val, dim=-1)
      out = torch.zeros_like(x_flat)
      for e in range(self.num_experts):
        token_ids, slot = torch.where(topk_idx == e)
        if token_ids.numel() == 0:
          continue
        y = self.experts[e](x_flat[token_ids])
        w = topk_w[token_ids, slot]
        out[token_ids] += w[:, None] * y

      return out.reshape(B, S, D)








In [4]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

Output: torch.Size([2, 8, 32])
Params: 16900


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (11.6ms)
  ✅ [2/4] Has router and experts (1.2ms)
  ✅ [3/4] Router logits shape (1.8ms)
  ✅ [4/4] Gradient flow (47.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (61.9ms total)
  Progress saved. Run status() to see your dashboard.

